# 5-fold cross-validation on Kaggle: c-ResUNet-light

The 212 trainval images are split into 5 folds stratified by the number of cells per image (`splits/green_cv{0..4}.json`,
written by `train/kfold_split.py`). For each fold, a network is trained on the other four folds (169–170 images) and predicts
the held-out fold with `last.pt` (epoch 100). `best.pt` is not used, because it is selected on the held-out fold itself.
Every trainval image thus receives a prediction from a network that has not seen it.

One run = one recipe on the 5 folds:

| `RECIPE` | options | output |
|---|---|---|
| `"light"` | `--small_rf` (adopted recipe) | `runs/cresunet_light_cv{k}/` |
| `"light_weight_maps"` | `--small_rf --weight_maps` (weight maps of Morelli et al., 2021, σ = 15 px, in the BCE) | `runs/cresunet_light_wm_cv{k}/` |

Each fold: training (100 epochs, no early stopping, seed 42) and predictions on the held-out fold (`pred_val.npz`).
About 57 min per fold, ~4.8 h per recipe; the two recipes can run as two parallel versions. The comparison of the recipes is computed locally from the `pred_val.npz` files.

Before running (the two input datasets are described in `kaggle/README.md`):
1. **Settings → Accelerator → GPU T4 x2**. Internet is not needed.
2. **Add Input**: the code dataset `cellseg-code` and the data dataset `fnc-green-data`.
3. Short check: `PROBE = True` (parameters cell) → **Run All**: fold 0, 1 epoch on 8 training and 4 validation images (~1 min).
4. Full run: `PROBE = False` → **Save Version → Save & Run All (Commit)**; the commit keeps running with the browser closed.
5. Download the output folders from the **Output** tab of the version into the repository's `runs/`.

In [ ]:
# 1) GPU available check
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> 'GPU T4 x2'"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Find code and data under /kaggle/input, copy the code to /kaggle/working (/kaggle/input is read-only)
import os
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def find_one(pattern):
    '''First path under INPUT matching the pattern (Kaggle can nest dataset folders in different ways).'''
    hits = sorted(INPUT.rglob(pattern))
    assert hits, f"not found under {INPUT}: {pattern} (were both datasets added with 'Add Input'?)"
    return hits[0]

CODE_SRC = find_one("models/c_res_unet_small_rf.py").parent.parent
IMAGES = find_one("green/trainval/images")
MASKS = find_one("cleaned_masks/trainval/Green/masks")
print("code:  ", CODE_SRC)
print("images:", IMAGES, "→", len(list(IMAGES.glob("*.png"))), "png (expected 212)")
print("masks: ", MASKS, "→", len(list(MASKS.glob("*.png"))), "png (expected 213: 128.png has no image)")

CODE = WORK / "code"
shutil.copytree(CODE_SRC, CODE, dirs_exist_ok=True)
os.chdir(CODE)
assert all((CODE / f"splits/green_cv{k}.json").exists() for k in range(5)), "splits/green_cv*.json missing from the code dataset"
print("working dir:", os.getcwd())

## 3) Parameters

- `RECIPE`: `"light"` or `"light_weight_maps"`.
- `PROBE = True`: short check; `PROBE = False`: full run.
- `FOLDS`: all five; fewer only to resume an interrupted run.

In [ ]:
RECIPE = "light"   # "light" or "light_weight_maps"
PROBE = True
FOLDS = [0, 1, 2, 3, 4]

RECIPES = {"light": ("cresunet_light_cv", "--small_rf"),
           "light_weight_maps": ("cresunet_light_wm_cv", "--small_rf --weight_maps")}
PREFIX, EXTRA = RECIPES[RECIPE]
OPTIONS = f"{EXTRA} --epochs 100 --patience 100"
if PROBE:
    FOLDS = [0]
    PREFIX = "probe_" + PREFIX
    OPTIONS = f"{EXTRA} --epochs 1 --patience 1 --train_limit 8 --val_limit 4"
print(RECIPE, "| folds", FOLDS, "| output runs/" + PREFIX + "{k}", "|", OPTIONS)

## 4) Training and predictions, fold by fold

The loop stops at the first error.

In [ ]:
for k in FOLDS:
    out = WORK / "runs" / f"{PREFIX}{k}"
    split = f"splits/green_cv{k}.json"
    !python -m train.training --images_dir "{IMAGES}" --masks_dir "{MASKS}" --split_json {split} --out_dir "{out}" --num_workers 4 {OPTIONS}
    !python -m evaluation.predict --ckpt "{out}/last.pt" --images_dir "{IMAGES}" --masks_dir "{MASKS}" --split_json {split} --subset val --out "{out}/pred_val.npz"
    assert (out / "pred_val.npz").exists(), f"fold {k} failed: see the messages above"
    print(f"=== fold {k} done")

In [ ]:
# 5) Summary: one row per fold
import json
import numpy as np
import pandas as pd

rows = []
for k in FOLDS:
    out = WORK / "runs" / f"{PREFIX}{k}"
    log = pd.read_csv(out / "log.csv")
    cfg = json.load(open(out / "config.json"))
    pred = np.load(out / "pred_val.npz")
    held = json.load(open(f"splits/green_cv{k}.json"))["val"]
    assert cfg["small_rf"] and list(pred["names"]) == held, f"fold {k}: wrong network or predicted images"
    rows.append({"fold": k, "epochs": len(log), "weight_maps": cfg["weight_maps"], "n_train": cfg["n_train"],
                 "held_out": len(held), "last_dice": log.val_dice_global.iloc[-1], "min_per_epoch": log.time_s.mean() / 60})
print(pd.DataFrame(rows).round(4).to_string(index=False))